<a href="https://colab.research.google.com/github/Nurdaylight/Study/blob/main/YandexAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import requests
from google.colab import userdata

# Retrieve API key from Secrets
api_key = userdata.get('Yandex_key')

def lookup_word(word, lang="en-ru"):
    url = "https://dictionary.yandex.net/api/v1/dicservice.json/lookup"
    params = {"key": api_key, "lang": lang, "text": word}

    response = requests.get(url, params=params)
    if response.status_code != 200:
        return f"Error: {response.status_code}"

    data = response.json()

    if not data.get('def'):
        print("No definitions found.")
        return

    print(f"--- Dictionary Entry for: {word.upper()} ---\n")

    for entry in data['def']:
        # Part of Speech (noun, verb, etc.)
        pos = entry.get('pos', 'unknown')
        print(f"Part of Speech: {pos}")

        # Transcription (how to pronounce it)
        ts = entry.get('ts')
        if ts: print(f"Transcription: [{ts}]")

        print("-" * 20)

        for tr in entry.get('tr', []):
            # Main Translation
            print(f"• Translation: {tr['text']}")

            # Synonyms (Russian words similar to the translation)
            syns = [s['text'] for s in tr.get('syn', [])]
            if syns:
                print(f"  Synonyms: {', '.join(syns)}")

            # Meanings (English context for this specific translation)
            means = [m['text'] for m in tr.get('mean', [])]
            if means:
                print(f"  Meanings (Context): {', '.join(means)}")

            # Examples (Phrases using the word)
            exs = tr.get('ex', [])
            if exs:
                print("  Examples:")
                for ex in exs:
                    ex_text = ex['text']
                    ex_tr = ex['tr'][0]['text'] if ex.get('tr') else ""
                    print(f"    - {ex_text} ({ex_tr})")
            print("")

# Run the function
lookup_word("screen")

--- Dictionary Entry for: SCREEN ---

Part of Speech: noun
Transcription: [skriːn]
--------------------
• Translation: экран
  Synonyms: дисплей, экран дисплея, экранчик
  Meanings (Context): display

• Translation: ширма
  Synonyms: завеса
  Meanings (Context): cover, veil

• Translation: сито
  Meanings (Context): sieve

• Translation: грохот
  Meanings (Context): crash

• Translation: заслон
  Meanings (Context): barrier

• Translation: меню
  Meanings (Context): menu

• Translation: сетка
  Meanings (Context): grid

• Translation: киноэкран
  Meanings (Context): movie screen

• Translation: растр
  Meanings (Context): raster

Part of Speech: verb
Transcription: [skriːn]
--------------------
• Translation: экранировать
  Meanings (Context): escape

• Translation: отсеивать
  Meanings (Context): sift

Part of Speech: adjective
Transcription: [skriːn]
--------------------
• Translation: экранный
  Meanings (Context): display

• Translation: сетчатый

Part of Speech: adverb
Transcripti

In [71]:
import requests

def get_russian_definition(word, api_key):
    params = {'key': api_key, 'lang': 'en-ru', 'text': word}
    res = requests.get("https://dictionary.yandex.net/api/v1/dicservice.json/lookup", params=params)

    if not res.ok: return f"Error: {res.status_code}"

    data = res.json().get('def', [])
    lines = [f"[{e.get('pos', '???')}] {', '.join(t['text'] for t in e['tr'])}" for e in data]

    return "\n".join(lines) if lines else "No definition found."

print(get_russian_definition("water", api_key))

[noun] вода, водоем, акватория, влага, водность, волны
[adjective] водяной
[verb] поливать, мочить


In [78]:
import requests

def get_detailed_russian_definition(word, api_key):
    # 'ui': 'en' ensures the part-of-speech labels are in English
    params = {'key': api_key, 'lang': 'en-ru', 'text': word, 'ui': 'en'}
    url = "https://dictionary.yandex.net/api/v1/dicservice.json/lookup"

    res = requests.get(url, params=params)
    if not res.ok:
        return f"Error: {res.status_code}"

    data = res.json().get('def', [])
    if not data:
        return "No definition found."

    output = []
    for entry in data:
        # 1. Get Transcription (e.g., [ˈwɔːtə])
        ts = f"/{entry['ts']}/" if 'ts' in entry else ""
        pos = entry.get('pos', '???')

        output.append(f"--- {entry['text']} {ts} [{pos}] ---")

        # 2. Process Translations
        for tr in entry.get('tr', []):
            # Extract gender (gen) if available (common for Russian nouns)
            gender = f"({tr['gen']})" if 'gen' in tr else ""
            line = f" • {tr['text']} {gender}"

            # 3. Add Synonyms if they exist
            if 'syn' in tr:
                syns = ", ".join([s['text'] for s in tr['syn']])
                line += f" (syn: {syns})"

            output.append(line)

            # 4. Add Examples (optional)
            for ex in tr.get('ex', [])[:1]: # Just the first example
                output.append(f"   Ex: {ex['text']} — {ex['tr'][0]['text']}")

    return "\n".join(output)

# Example Usage
print(get_detailed_russian_definition("water", api_key))

--- water /ˈwɔːtə/ [noun] ---
 • вода (ж) (syn: водичка)
 • водоем (м)
 • акватория (ж)
 • влага (ж)
 • водность (ж)
 • волны (ж)
--- water /ˈwɔːtə/ [adjective] ---
 • водяной  (syn: водный, водопроводный, водонапорный)
--- water /ˈwɔːtə/ [verb] ---
 • поливать  (syn: полить, поить)
 • мочить  (syn: намочить)


In [80]:
import requests
import json

# 1. Define the extraction function
def extract_all_metadata(res_json):
    extracted_data = []
    for definition in res_json.get('def', []):
        entry = {
            "source": definition.get('text'),
            "transcription": definition.get('ts'),
            "pos": definition.get('pos'),
            "results": []
        }
        for tr in definition.get('tr', []):
            translation = {
                "word": tr.get('text'),
                "gender": tr.get('gen'),      # e.g., 'f' (feminine)
                "aspect": tr.get('asp'),      # e.g., 'perf' (perfective)
                "number": tr.get('num'),      # e.g., 'pl' (plural)
                "synonyms": [s.get('text') for s in tr.get('syn', [])],
                "meanings": [m.get('text') for m in tr.get('mean', [])],
                "examples": [
                    {"src": e.get('text'), "dst": e.get('tr', [{}])[0].get('text')}
                    for e in tr.get('ex', [])
                ]
            }
            entry["results"].append(translation)
        extracted_data.append(entry)
    return extracted_data

# 2. Make the API Call
api_key = api_key
word = 'water'
params = {'key': api_key, 'lang': 'en-ru', 'text': word}
url = "https://dictionary.yandex.net/api/v1/dicservice.json/lookup"

response = requests.get(url, params=params)

# 3. Call the function and print the clean data
if response.ok:
    full_json = response.json()
    formatted_data = extract_all_metadata(full_json)

    # Pretty print the result
    print(json.dumps(formatted_data, indent=4, ensure_ascii=False))
else:
    print(f"Error: {response.status_code}")

[
    {
        "source": "water",
        "transcription": "ˈwɔːtə",
        "pos": "noun",
        "results": [
            {
                "word": "вода",
                "gender": "ж",
                "aspect": null,
                "number": null,
                "synonyms": [
                    "водичка"
                ],
                "meanings": [
                    "water supply"
                ],
                "examples": []
            },
            {
                "word": "водоем",
                "gender": "м",
                "aspect": null,
                "number": null,
                "synonyms": [],
                "meanings": [
                    "reservoir"
                ],
                "examples": []
            },
            {
                "word": "акватория",
                "gender": "ж",
                "aspect": null,
                "number": null,
                "synonyms": [],
                "meanings": [
                    "water